In [27]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score, \
    max_error, mean_poisson_deviance, mean_gamma_deviance, mean_tweedie_deviance, mean_absolute_percentage_error
from catboost import CatBoostRegressor, Pool
import matplotlib.pyplot as plt
import optuna
from prophet import Prophet


### Reading the dataset as CSV

In [28]:
df = pd.read_csv('../input/electricity-load-forecasting/continuous dataset.csv')
df.head()

,datetime,nat_demand,T2M_toc,QV2M_toc,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,T2M_dav,QV2M_dav,TQL_dav,W2M_dav,Holiday_ID,holiday,school
0,2015-01-03 01:00:00,970.3450,25.865259,0.018576,0.016174,21.850546,23.482446,0.017272,0.001855,10.328949,22.662134,0.016562,0.096100,5.364148,0,0,0
1,2015-01-03 02:00:00,912.1755,25.899255,0.018653,0.016418,22.166944,23.399255,0.017265,0.001327,10.681517,22.578943,0.016509,0.087646,5.572471,0,0,0
2,2015-01-03 03:00:00,900.2688,25.937280,0.018768,0.015480,22.454911,23.343530,0.017211,0.001428,10.874924,22.531030,0.016479,0.078735,5.871184,0,0,0
3,2015-01-03 04:00:00,889.9538,25.957544,0.018890,0.016273,22.110481,23.238794,0.017128,0.002599,10.518620,22.512231,0.016487,0.068390,5.883621,0,0,0
4,2015-01-03 05:00:00,893.6865,25.973840,0.018981,0.017281,21.186089,23.075403,0.017059,0.001729,9.733589,22.481653,0.016456,0.064362,5.611724,0,0,0


### Splitting the dataset into train and test sets

In [29]:
train_df = df[df['datetime'] < "2019-01-01"]
train_df.head()

,datetime,nat_demand,T2M_toc,QV2M_toc,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,T2M_dav,QV2M_dav,TQL_dav,W2M_dav,Holiday_ID,holiday,school
0,2015-01-03 01:00:00,970.3450,25.865259,0.018576,0.016174,21.850546,23.482446,0.017272,0.001855,10.328949,22.662134,0.016562,0.096100,5.364148,0,0,0
1,2015-01-03 02:00:00,912.1755,25.899255,0.018653,0.016418,22.166944,23.399255,0.017265,0.001327,10.681517,22.578943,0.016509,0.087646,5.572471,0,0,0
2,2015-01-03 03:00:00,900.2688,25.937280,0.018768,0.015480,22.454911,23.343530,0.017211,0.001428,10.874924,22.531030,0.016479,0.078735,5.871184,0,0,0
3,2015-01-03 04:00:00,889.9538,25.957544,0.018890,0.016273,22.110481,23.238794,0.017128,0.002599,10.518620,22.512231,0.016487,0.068390,5.883621,0,0,0
4,2015-01-03 05:00:00,893.6865,25.973840,0.018981,0.017281,21.186089,23.075403,0.017059,0.001729,9.733589,22.481653,0.016456,0.064362,5.611724,0,0,0


In [30]:
test_df = df[df['datetime'] >= "2019-01-01"]
test_df.head()

,datetime,nat_demand,T2M_toc,QV2M_toc,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,T2M_dav,QV2M_dav,TQL_dav,W2M_dav,Holiday_ID,holiday,school
35015,2019-01-01 00:00:00,1027.9238,24.885645,0.017164,0.007330,22.597570,23.682520,0.016592,0.018089,12.133071,22.143457,0.016005,0.072266,6.110409,1,1,0
35016,2019-01-01 01:00:00,1006.3692,24.839471,0.017123,0.008972,23.223214,23.519159,0.016452,0.014172,12.098985,22.066034,0.015872,0.079041,6.145190,1,1,0
35017,2019-01-01 02:00:00,990.4364,24.862939,0.017205,0.011501,23.223043,23.402002,0.016335,0.012352,12.282677,21.980127,0.015694,0.080963,6.233335,1,1,0
35018,2019-01-01 03:00:00,975.8917,24.886011,0.017333,0.012527,22.834583,23.300073,0.016196,0.007969,12.509328,21.886011,0.015506,0.080414,6.368042,1,1,0
35019,2019-01-01 04:00:00,949.1626,24.932092,0.017439,0.012455,22.174321,23.166467,0.016096,0.007296,11.912222,21.838342,0.015379,0.078094,6.269188,1,1,0


### Feature Engineering

In [31]:
windows = [12, 24, 128]
for column in train_df.columns:
    if column == 'nat_demand':
        for window in windows:
            train_df[f"{column}_lag_{window}"] = train_df[column].shift(window)
            train_df[f"{column}_ma_mean{window}"] = train_df[column].rolling(window).mean()
            train_df[f"{column}_std_std{window}"] = train_df[column].rolling(window).std()
            train_df[f"{column}_ewm_std{window}"] = train_df[column].ewm(window).std()
            train_df[f"{column}_ewm_mean{window}"] = train_df[column].ewm(window).mean()
            
            
    if column != 'datetime' and column != 'holiday' and column != 'school' and column != 'Holiday_ID' and column != 'nat_demand':
        for window in windows:
            train_df[f"{column}_lag_{window}"] = train_df[column].shift(window)
            train_df[f"{column}_ma_mean{window}"] = train_df[column].rolling(window).mean()
            train_df[f"{column}_std_std{window}"] = train_df[column].rolling(window).std()
            train_df[f"{column}_ewm_std{window}"] = train_df[column].ewm(window).std()
            train_df[f"{column}_ewm_mean{window}"] = train_df[column].ewm(window).mean()
            train_df[f"{column}_min_max{window}"] = (train_df[column] -train_df[column].rolling(window).min()) / (train_df[column].rolling(window).max() - train_df[column].rolling(window).min())
            train_df[f"{column}_median{window}"] = train_df[column].rolling(window).median()
            train_df[f"{column}_skew{window}"] = train_df[column].rolling(window).skew()
            train_df[f"{column}_kurt{window}"] = train_df[column].rolling(window).kurt()
            train_df[f"{column}_p50{window}"] = train_df[column].rolling(window).quantile(0.5)

train_df.dropna(inplace=True)

/tmp/ipykernel_35/1200432947.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[f"{column}_lag_{window}"] = train_df[column].shift(window)
/tmp/ipykernel_35/1200432947.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[f"{column}_ma_mean{window}"] = train_df[column].rolling(window).mean()
/tmp/ipykernel_35/1200432947.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in

In [32]:
train_df

,datetime,nat_demand,T2M_toc,QV2M_toc,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,...,W2M_dav_lag_128,W2M_dav_ma_mean128,W2M_dav_std_std128,W2M_dav_ewm_std128,W2M_dav_ewm_mean128,W2M_dav_min_max128,W2M_dav_median128,W2M_dav_skew128,W2M_dav_kurt128,W2M_dav_p50128
128,2015-01-08 09:00:00,1275.1120,27.571802,0.016819,0.026474,33.521500,26.884302,0.017131,0.062561,18.893116,...,5.364148,6.199112,1.491481,1.390388,6.439521,0.973497,6.394866,-0.567548,-0.519360,6.394866
129,2015-01-08 10:00:00,1348.9510,28.399377,0.016773,0.025772,33.816953,27.993127,0.017017,0.058289,19.903031,...,5.572471,6.220771,1.502398,1.397702,6.462729,0.982404,6.464252,-0.575322,-0.525737,6.464252
130,2015-01-08 11:00:00,1365.6284,29.004724,0.016819,0.031982,33.338703,28.785974,0.017048,0.055099,19.899943,...,5.871184,6.240072,1.513699,1.404431,6.485517,0.981865,6.485280,-0.579291,-0.539667,6.485280
131,2015-01-08 12:00:00,1343.4717,29.328638,0.016871,0.048721,32.322581,29.297388,0.017054,0.044006,19.667262,...,5.883621,6.260305,1.526154,1.412813,6.509520,1.000000,6.503291,-0.580459,-0.552839,6.503291
132,2015-01-08 13:00:00,1371.5959,29.372705,0.016870,0.065063,31.365933,29.528955,0.016962,0.046646,19.495408,...,5.611724,6.284000,1.539492,1.423564,6.535190,1.000000,6.522611,-0.582098,-0.556282,6.522611
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35010,2018-12-31 19:00:00,1278.6240,25.926416,0.017542,0.006657,23.312495,24.371729,0.016635,0.002964,10.364903,...,6.980621,4.765239,1.480682,1.519725,4.903959,0.867298,4.849127,-0.864839,0.903443,4.849127
35011,2018-12-31 20:00:00,1241.6289,25.626184,0.017188,0.007753,23.396554,24.055872,0.016654,0.005663,10.329677,...,6.977380,4.760650,1.474671,1.519445,4.915479,0.865830,4.849127,-0.882278,0.927637,4.849127
35012,2018-12-31 21:00:00,1193.0323,25.398706,0.017038,0.003750,23.285532,23.875269,0.016672,0.007523,10.568211,...,7.005403,4.753688,1.466068,1.517206,4.924772,0.824538,4.849127,-0.905845,0.965076,4.849127
35013,2018-12-31 22:00:00,1137.6514,25.179926,0.017070,0.002913,22.767433,23.804926,0.016696,0.013668,11.376140,...,7.024324,4.746265,1.456857,1.514685,4.933682,0.818517,4.849127,-0.931936,1.005247,4.849127


In [33]:
windows = [12, 24, 128]
for column in test_df.columns:
    if column == 'nat_demand':
        for window in windows:
        
            test_df[f"{column}_lag_{window}"] = test_df[column].shift(window)
            test_df[f"{column}_lag_{window}"] = test_df[column].shift(window)
            test_df[f"{column}_ma_mean{window}"] = test_df[column].rolling(window).mean()
            test_df[f"{column}_std_std{window}"] = test_df[column].rolling(window).std()
            test_df[f"{column}_ewm_std{window}"] = test_df[column].ewm(window).std()
            test_df[f"{column}_ewm_mean{window}"] = test_df[column].ewm(window).mean()
            
    if column != 'datetime' and column != 'holiday' and column != 'school' and column != 'Holiday_ID' and column != 'nat_demand':
        for window in windows:
            test_df[f"{column}_lag_{window}"] = test_df[column].shift(window)
            test_df[f"{column}_ma_mean{window}"] = test_df[column].rolling(window).mean()
            test_df[f"{column}_std_std{window}"] = test_df[column].rolling(window).std()
            test_df[f"{column}_ewm_std{window}"] = test_df[column].ewm(window).std()
            test_df[f"{column}_ewm_mean{window}"] = test_df[column].ewm(window).mean()
            test_df[f"{column}_min_max{window}"] = (test_df[column] -test_df[column].rolling(window).min()) / (test_df[column].rolling(window).max() - test_df[column].rolling(window).min())
            test_df[f"{column}_median{window}"] = test_df[column].rolling(window).median()
            test_df[f"{column}_skew{window}"] = test_df[column].rolling(window).skew()
            test_df[f"{column}_kurt{window}"] = test_df[column].rolling(window).kurt()
            test_df[f"{column}_p50{window}"] = test_df[column].rolling(window).quantile(0.5)

test_df.dropna(inplace=True)

/tmp/ipykernel_35/4250250253.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[f"{column}_lag_{window}"] = test_df[column].shift(window)
/tmp/ipykernel_35/4250250253.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[f"{column}_lag_{window}"] = test_df[column].shift(window)
/tmp/ipykernel_35/4250250253.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentati

In [34]:
test_df

,datetime,nat_demand,T2M_toc,QV2M_toc,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,...,W2M_dav_lag_128,W2M_dav_ma_mean128,W2M_dav_std_std128,W2M_dav_ewm_std128,W2M_dav_ewm_mean128,W2M_dav_min_max128,W2M_dav_median128,W2M_dav_skew128,W2M_dav_kurt128,W2M_dav_p50128
35143,2019-01-06 08:00:00,1010.4562,26.479791,0.016606,0.024277,25.745778,25.511041,0.016995,0.017456,11.846568,...,6.110409,4.575148,1.511192,1.469350,4.434729,0.584200,4.869911,-0.794132,0.105066,4.869911
35144,2019-01-06 09:00:00,1050.7971,27.411096,0.016399,0.031555,27.262736,27.270471,0.017185,0.014236,14.011013,...,6.145190,4.560622,1.504904,1.460425,4.432914,0.540201,4.853817,-0.783848,0.116907,4.853817
35145,2019-01-06 10:00:00,1100.5710,28.260309,0.016414,0.027863,27.282461,28.541559,0.017299,0.011993,14.315570,...,6.233335,4.543838,1.498066,1.452007,4.428696,0.507997,4.842220,-0.771877,0.126300,4.842220
35146,2019-01-06 11:00:00,1154.8653,28.903589,0.016452,0.027283,26.760379,29.512964,0.017390,0.011295,14.150185,...,6.368042,4.523386,1.490818,1.445092,4.420503,0.454270,4.822929,-0.757241,0.127051,4.822929
35147,2019-01-06 12:00:00,1168.6314,29.288812,0.016428,0.026230,26.304043,30.148187,0.017473,0.012177,13.900855,...,6.269188,4.499522,1.487095,1.442391,4.406005,0.368330,4.788671,-0.732426,0.098339,4.788671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48043,2020-06-26 20:00:00,1128.5592,27.246545,0.020303,0.055511,9.289304,25.715295,0.019746,0.121552,1.990773,...,1.519354,2.434515,0.858886,1.080783,2.716574,0.585725,2.437952,-0.269729,-0.461723,2.437952
48044,2020-06-26 21:00:00,1112.7488,27.099573,0.020395,0.053848,9.837504,25.552698,0.019632,0.153870,2.094459,...,2.021437,2.438377,0.858125,1.076730,2.715018,0.605266,2.450098,-0.283265,-0.447758,2.450098
48045,2020-06-26 22:00:00,1081.5680,26.971155,0.020448,0.057251,10.262464,25.393030,0.019518,0.144531,2.396369,...,2.240761,2.442752,0.858537,1.072575,2.715682,0.683506,2.457710,-0.297688,-0.446453,2.457710
48046,2020-06-26 23:00:00,1041.6240,26.867487,0.020464,0.064178,10.326567,25.258112,0.019403,0.108063,2.720871,...,1.974162,2.451845,0.859699,1.069054,2.718957,0.776166,2.463881,-0.323192,-0.444398,2.463881


In [35]:
X_train, X_test = train_df.drop(columns = ['nat_demand','datetime']), test_df.drop(columns = ['nat_demand','datetime'])
y_train, y_test = train_df['nat_demand'], test_df['nat_demand']

### Feature Scaling

In [36]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Training the XGBoost Model

In [37]:
dtrain = xgb.DMatrix(X_train_scaled, label=y_train, feature_names=X_train.columns.to_list())
dtest = xgb.DMatrix(X_test_scaled, label=y_test, feature_names=X_test.columns.to_list())

In [38]:
def objective(trial):
    params = {
        'objective': 'reg:squarederror',
        'tree_method': 'gpu_hist',
        'gpu_id': 0,
        'eta': trial.suggest_float('eta', 0.001, 0.5),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_child_weight': trial.suggest_float('min_child_weight', 0.1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'lambda': trial.suggest_float('lambda', 0.0, 2.0),
        'alpha': trial.suggest_float('alpha', 0.0, 2.0),
        'nthread': -1,
        'seed': 42
    }

    num_round = 100
    model = xgb.train(params, dtrain, num_round)

    predictions = model.predict(dtest)
    rmse = mean_squared_error(y_test, predictions, squared=False)
    return rmse

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)

In [40]:
best_params = study.best_params
best_params['tree_method'] = 'gpu_hist'
best_params['gpu_id'] = 0
print("Best params:", best_params)
xgb_best_params = best_params

Best params: {'eta': 0.06215152871099107, 'max_depth': 8, 'min_child_weight': 3.4113108913529677, 'subsample': 0.6838710401875765, 'colsample_bytree': 0.9377121210038669, 'gamma': 3.117910818512726, 'lambda': 0.07566791864622047, 'alpha': 0.7494047352013485, 'tree_method': 'gpu_hist', 'gpu_id': 0}


In [41]:
xgb_model = xgb.train(best_params, dtrain, 100)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [10:33:24] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


In [42]:
predictions = xgb_predictions = xgb_model.predict(dtest)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [10:33:25] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


### Performance of XGBoost Model

In [43]:
rmse_test = mean_squared_error(y_test, predictions, squared=False)
mae_test = mean_absolute_error(y_test, predictions)
r2_test = r2_score(y_test, predictions)
explained_variance = explained_variance_score(y_test, predictions)
max_err = max_error(y_test, predictions)
poisson_deviance = mean_poisson_deviance(y_test, predictions)
gamma_deviance = mean_gamma_deviance(y_test, predictions)
tweedie_deviance = mean_tweedie_deviance(y_test, predictions)
mape = mean_absolute_percentage_error(y_test, predictions)
print(f"RMSE on test data: {rmse_test}")
print(f"MAE on test data: {mae_test}")
print("Mean Absolute Percentage Error:", mape)
print(f"R-squared on test data: {r2_test}")
print(f"Explained Variance on test data: {explained_variance}")
print(f"Max Error on test data: {max_err}")
print(f"Mean Poisson Deviance on test data: {poisson_deviance}")
print(f"Mean Gamma Deviance on test data: {gamma_deviance}")
print(f"Mean Tweedie Deviance on test data: {tweedie_deviance}")

RMSE on test data: 49.91707253089567
MAE on test data: 32.16243893097243
Mean Absolute Percentage Error: 0.028334063037420455
R-squared on test data: 0.9295856067168204
Explained Variance on test data: 0.9336286640547085
Max Error on test data: 1219.21228515625
Mean Poisson Deviance on test data: 2.195241493282646
Mean Gamma Deviance on test data: 0.0021604843731121556
Mean Tweedie Deviance on test data: 2491.714130054699


### Sensitivity test of XGBoost Model

In [ ]:
def remove_random_data(X, y, removal_percentage=0.1):
    num_points_to_remove = int(len(X) * removal_percentage)
    indices_to_remove = np.random.choice(len(X), num_points_to_remove, replace=False)

    X_filtered = np.delete(X, indices_to_remove, axis=0)
    y_filtered = np.delete(y, indices_to_remove)

    return X_filtered, y_filtered

removal_rates = list(range(10, 51, 10))
rmse_scores = []
mae_scores = []
r2_scores = []
explained_variance_scores = []
max_err_scores = []
poisson_deviance_scores = []
gamma_deviance_scores = []
tweedie_deviance_scores = []
mape_scores = []

for rate in removal_rates:
    X_train_filtered, y_train_filtered = remove_random_data(X_train_scaled, y_train.values, removal_percentage=rate / 100)
    X_test_filtered, y_test_filtered = remove_random_data(X_test_scaled, y_test.values, removal_percentage=rate / 100)

    dtrain_filtered = xgb.DMatrix(X_train_filtered, label=y_train_filtered, feature_names=X_train.columns.to_list())

    num_round = 100
    model_filtered = xgb.train(best_params, dtrain_filtered, num_round)

    dtest_filtered = xgb.DMatrix(X_test_filtered, label=y_test_filtered, feature_names=X_test.columns.to_list())

    predictions_filtered = model_filtered.predict(dtest_filtered)

    rmse_scores.append(mean_squared_error(y_test_filtered, predictions_filtered, squared=False))
    mae_scores.append(mean_absolute_error(y_test_filtered, predictions_filtered))
    r2_scores.append(r2_score(y_test_filtered, predictions_filtered))
    explained_variance_scores.append(explained_variance_score(y_test_filtered, predictions_filtered))
    max_err_scores.append(max_error(y_test_filtered, predictions_filtered))
    poisson_deviance_scores.append(mean_poisson_deviance(y_test_filtered, predictions_filtered))
    gamma_deviance_scores.append(mean_gamma_deviance(y_test_filtered, predictions_filtered))
    tweedie_deviance_scores.append(mean_tweedie_deviance(y_test_filtered, predictions_filtered))
    mape_scores.append(mean_absolute_percentage_error(y_test_filtered, predictions_filtered))

### Training the Prophet Model

In [ ]:
prophet_train_df = pd.DataFrame({'ds': train_df['datetime'], 'y': train_df['nat_demand']})

prophet_model = Prophet()

num_columns2 = len(train_df.columns)
for i in range(2, num_columns2):  
    regressor_name = train_df.columns[i]
    prophet_model.add_regressor(regressor_name)
    prophet_train_df[regressor_name] = train_df[regressor_name]

prophet_model.fit(prophet_train_df)

prophet_future = pd.DataFrame({'ds': test_df['datetime']})

for i in range(2, num_columns2):
    regressor_name = train_df.columns[i]
    prophet_future[regressor_name] = test_df[regressor_name]

prophet_forecast = prophet_model.predict(prophet_future)
prophet_predictions = prophet_forecast['yhat']

In [47]:
prophet_future

,ds,T2M_toc,QV2M_toc,TQL_toc,W2M_toc,T2M_san,QV2M_san,TQL_san,W2M_san,T2M_dav,...,W2M_dav_lag_128,W2M_dav_ma_mean128,W2M_dav_std_std128,W2M_dav_ewm_std128,W2M_dav_ewm_mean128,W2M_dav_min_max128,W2M_dav_median128,W2M_dav_skew128,W2M_dav_kurt128,W2M_dav_p50128
35143,2019-01-06 08:00:00,26.479791,0.016606,0.024277,25.745778,25.511041,0.016995,0.017456,11.846568,23.378229,...,6.110409,4.575148,1.511192,1.469350,4.434729,0.584200,4.869911,-0.794132,0.105066,4.869911
35144,2019-01-06 09:00:00,27.411096,0.016399,0.031555,27.262736,27.270471,0.017185,0.014236,14.011013,24.934534,...,6.145190,4.560622,1.504904,1.460425,4.432914,0.540201,4.853817,-0.783848,0.116907,4.853817
35145,2019-01-06 10:00:00,28.260309,0.016414,0.027863,27.282461,28.541559,0.017299,0.011993,14.315570,26.330621,...,6.233335,4.543838,1.498066,1.452007,4.428696,0.507997,4.842220,-0.771877,0.126300,4.842220
35146,2019-01-06 11:00:00,28.903589,0.016452,0.027283,26.760379,29.512964,0.017390,0.011295,14.150185,27.434839,...,6.368042,4.523386,1.490818,1.445092,4.420503,0.454270,4.822929,-0.757241,0.127051,4.822929
35147,2019-01-06 12:00:00,29.288812,0.016428,0.026230,26.304043,30.148187,0.017473,0.012177,13.900855,28.148187,...,6.269188,4.499522,1.487095,1.442391,4.406005,0.368330,4.788671,-0.732426,0.098339,4.788671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48043,2020-06-26 20:00:00,27.246545,0.020303,0.055511,9.289304,25.715295,0.019746,0.121552,1.990773,23.746545,...,1.519354,2.434515,0.858886,1.080783,2.716574,0.585725,2.437952,-0.269729,-0.461723,2.437952
48044,2020-06-26 21:00:00,27.099573,0.020395,0.053848,9.837504,25.552698,0.019632,0.153870,2.094459,23.693323,...,2.021437,2.438377,0.858125,1.076730,2.715018,0.605266,2.450098,-0.283265,-0.447758,2.450098
48045,2020-06-26 22:00:00,26.971155,0.020448,0.057251,10.262464,25.393030,0.019518,0.144531,2.396369,23.658655,...,2.240761,2.442752,0.858537,1.072575,2.715682,0.683506,2.457710,-0.297688,-0.446453,2.457710
48046,2020-06-26 23:00:00,26.867487,0.020464,0.064178,10.326567,25.258112,0.019403,0.108063,2.720871,23.601862,...,1.974162,2.451845,0.859699,1.069054,2.718957,0.776166,2.463881,-0.323192,-0.444398,2.463881


### Performance of Prophet Model

In [48]:
rmse_test = mean_squared_error(y_test, prophet_predictions, squared=False)
mae_test = mean_absolute_error(y_test, prophet_predictions)
r2_test = r2_score(y_test, prophet_predictions)
explained_variance = explained_variance_score(y_test, prophet_predictions)
max_err = max_error(y_test, prophet_predictions)
poisson_deviance = mean_poisson_deviance(y_test, prophet_predictions)
gamma_deviance = mean_gamma_deviance(y_test, prophet_predictions)
tweedie_deviance = mean_tweedie_deviance(y_test, prophet_predictions)
mape = mean_absolute_percentage_error(y_test, prophet_predictions)
print(f"RMSE on test data: {rmse_test}")
print(f"MAE on test data: {mae_test}")
print("Mean Absolute Percentage Error:", mape)
print(f"R-squared on test data: {r2_test}")
print(f"Explained Variance on test data: {explained_variance}")
print(f"Max Error on test data: {max_err}")
print(f"Mean Poisson Deviance on test data: {poisson_deviance}")
print(f"Mean Gamma Deviance on test data: {gamma_deviance}")
print(f"Mean Tweedie Deviance on test data: {tweedie_deviance}")

RMSE on test data: 42.11920139268886
MAE on test data: 30.21405696302087
Mean Absolute Percentage Error: 0.026843995250630436
R-squared on test data: 0.949867024376715
Explained Variance on test data: 0.9539341621775818
Max Error on test data: 622.6178874028507
Mean Poisson Deviance on test data: 1.636181707571133
Mean Gamma Deviance on test data: 0.0017262527532048348
Mean Tweedie Deviance on test data: 1774.027125957883


### Ensembling XGBoost and Prophet Models

In [49]:
ensemble_predictions_filtered = (0.7 * prophet_predictions) + (0.3 * xgb_predictions)

meta_model = CatBoostRegressor(iterations=100, depth=6, learning_rate=0.1, loss_function='RMSE')
meta_model.fit(np.column_stack((prophet_predictions, xgb_predictions)), y_test)
    
meta_predictions = meta_model.predict(np.column_stack((prophet_predictions, xgb_predictions)))

0:	learn: 171.1304354	total: 3.03ms	remaining: 300ms
1:	learn: 155.6293970	total: 5.16ms	remaining: 253ms
2:	learn: 141.8249052	total: 7.01ms	remaining: 227ms
3:	learn: 129.4781087	total: 8.76ms	remaining: 210ms
4:	learn: 118.2875431	total: 10.5ms	remaining: 199ms
5:	learn: 108.2538113	total: 12.4ms	remaining: 194ms
6:	learn: 99.4066116	total: 14.2ms	remaining: 188ms
7:	learn: 91.3572160	total: 15.8ms	remaining: 182ms
8:	learn: 84.1705113	total: 17.6ms	remaining: 178ms
9:	learn: 77.8333192	total: 19.2ms	remaining: 173ms
10:	learn: 72.1523874	total: 21ms	remaining: 170ms
11:	learn: 67.1509286	total: 22.8ms	remaining: 167ms
12:	learn: 62.8349443	total: 24.6ms	remaining: 164ms
13:	learn: 58.9861551	total: 26.3ms	remaining: 162ms
14:	learn: 55.4671212	total: 28.1ms	remaining: 159ms
15:	learn: 52.4593208	total: 30ms	remaining: 158ms
16:	learn: 49.8279709	total: 31.8ms	remaining: 155ms
17:	learn: 47.6548321	total: 33.6ms	remaining: 153ms
18:	learn: 45.6036430	total: 35.4ms	remaining: 151ms
1

### Performance of Meta Model

In [50]:
rmse_test = mean_squared_error(y_test, meta_predictions, squared=False)
mae_test = mean_absolute_error(y_test, meta_predictions)
r2_test = r2_score(y_test, meta_predictions)
explained_variance = explained_variance_score(y_test, meta_predictions)
max_err = max_error(y_test, meta_predictions)
poisson_deviance = mean_poisson_deviance(y_test, meta_predictions)
gamma_deviance = mean_gamma_deviance(y_test, meta_predictions)
tweedie_deviance = mean_tweedie_deviance(y_test, meta_predictions)
mape = mean_absolute_percentage_error(y_test, meta_predictions)
print(f"RMSE on test data: {rmse_test}")
print(f"MAE on test data: {mae_test}")
print("Mean Absolute Percentage Error:", mape)
print(f"R-squared on test data: {r2_test}")
print(f"Explained Variance on test data: {explained_variance}")
print(f"Max Error on test data: {max_err}")
print(f"Mean Poisson Deviance on test data: {poisson_deviance}")
print(f"Mean Gamma Deviance on test data: {gamma_deviance}")
print(f"Mean Tweedie Deviance on test data: {tweedie_deviance}")

RMSE on test data: 31.992110749898302
MAE on test data: 23.397295199556357
Mean Absolute Percentage Error: 0.01990701827476235
R-squared on test data: 0.9710766218473045
Explained Variance on test data: 0.9710766218666445
Max Error on test data: 229.20526987641642
Mean Poisson Deviance on test data: 0.8798764134816955
Mean Gamma Deviance on test data: 0.0007865935713764257
Mean Tweedie Deviance on test data: 1023.4951502337584


### Sensitivity test of Meta Model

In [ ]:
def remove_random_df(train_df_r, test_df_r, removal_percentage=0.1):
    num_points_to_remove_train = int(len(train_df_r) * removal_percentage)
    indices_to_remove_train = np.random.choice(len(train_df_r), num_points_to_remove_train, replace=False)

    num_points_to_remove_test = int(len(test_df_r) * removal_percentage)
    indices_to_remove_test = np.random.choice(len(test_df_r), num_points_to_remove_test, replace=False)

    train_df_filtered = train_df_r.drop(train_df_r.index[indices_to_remove_train]).reset_index(drop=True)
    test_df_filtered = test_df_r.drop(test_df_r.index[indices_to_remove_test]).reset_index(drop=True)

    return train_df_filtered, test_df_filtered

removal_rates = list(range(10, 51, 10))
rmse_scores = []

ensemble_rmse_scores = []
meta_rmse_scores = []

for rate in removal_rates:
    train_df_filtered, test_df_filtered = remove_random_df(train_df, test_df, removal_percentage=rate / 100)
    X_train_filtered, X_test_filtered = train_df_filtered.drop(columns = ['nat_demand','datetime']), test_df_filtered.drop(columns = ['nat_demand','datetime'])
    y_train_filtered, y_test_filtered = train_df_filtered['nat_demand'], test_df_filtered['nat_demand']
    
    X_train_filtered = scaler.fit_transform(X_train_filtered)
    X_test_filtered = scaler.transform(X_test_filtered)

    xgb_model_filtered = xgb.XGBRegressor(**xgb_best_params)  # Replace xgb_params with your XGBoost parameters
    xgb_model_filtered.fit(X_train_filtered, y_train_filtered)
    xgb_predictions_filtered = xgb_model_filtered.predict(X_test_filtered)
    
        #Propohet model
    prophet_train_df = pd.DataFrame({'ds': train_df_filtered['datetime'], 'y': train_df_filtered['nat_demand']})
    prophet_model = Prophet()

    num_columns2 = len(train_df_filtered.columns)
    for i in range(2, num_columns2):  # Assuming regressors start from the third column
        regressor_name = train_df_filtered.columns[i]
        prophet_model.add_regressor(regressor_name)
        prophet_train_df[regressor_name] = train_df_filtered[regressor_name]

    prophet_model.fit(prophet_train_df)

    prophet_future = pd.DataFrame({'ds': test_df_filtered['datetime']})

    for i in range(2, num_columns2):
        regressor_name = train_df_filtered.columns[i]
        prophet_future[regressor_name] = test_df_filtered[regressor_name]

    prophet_forecast = prophet_model.predict(prophet_future)


    prophet_predictions_filtered = prophet_forecast['yhat']
    ensemble_predictions_filtered = (0.7 * prophet_predictions_filtered) + (0.3 * xgb_predictions_filtered)
    

    meta_model = CatBoostRegressor(iterations=100, depth=6, learning_rate=0.1, loss_function='RMSE')
    meta_model.fit(np.column_stack((prophet_predictions_filtered, xgb_predictions_filtered)), y_test_filtered)
    
    meta_predictions_filtered = meta_model.predict(np.column_stack((prophet_predictions_filtered, xgb_predictions_filtered)))
    
    rmse_scores.append(mean_squared_error(test_df_filtered['nat_demand'], prophet_predictions_filtered, squared=False))
    
    ensemble_rmse_scores.append(mean_squared_error(y_test_filtered, ensemble_predictions_filtered, squared=False))
    meta_rmse_scores.append(mean_squared_error(y_test_filtered, meta_predictions_filtered, squared=False))

In [60]:
for r_rate , rmse_score in zip(removal_rates, meta_rmse_scores):
    print(f"Removal Rate {r_rate} | RMSE Score: {rmse_score}")

Removal Rate 10 | RMSE Score: 31.911159333066706
Removal Rate 20 | RMSE Score: 32.26838560126398
Removal Rate 30 | RMSE Score: 32.57341707983714
Removal Rate 40 | RMSE Score: 32.507730200225986
Removal Rate 50 | RMSE Score: 32.003275897010106
